<h2>Conceptual understanding</h2>

1. "Love" vs "love":
They are treated as different tokens due to case sensitivity. Normalization (lowercasing) ensures consistency.

2. Without stopword removal:
Models may focus on irrelevant frequent words, reducing performance and increasing noise.

3. When removing stopwords is harmful:
- Sentiment analysis ("not good" loses meaning if "not" is removed)
- Question answering ("who", "what", "where" are important)

4. Stemming vs Lemmatization:
Stemming cuts words crudely (running → runn)
Lemmatization converts to proper root form (running → run)

<h2>import necessary libraries</h2>

In [1]:
import re
from collections import Counter

<h2>Preprocessing Function</h2>

In [10]:
def preprocess_text(text):
    # Handle invalid / empty input
    if not isinstance(text, str) or text.strip() == "":
        return [], ""

    # Remove URLs & emails
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)

    # Remove emojis / non-ascii
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Convert to lowercase
    text = text.lower()

    # Handle repeated characters (soooo → so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Edge case: only symbols/emojis originally
    if text == "":
        return [], ""

    tokens = text.split()

    # Remove short tokens (<=2) except meaningful ones
    tokens = [w for w in tokens if len(w) > 2 or w in ['no', 'not']]

    clean_sentence = " ".join(tokens)

    return tokens, clean_sentence

<h2>Test data</h2>

In [11]:
sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this",
    
    # Extra test cases (for robustness)
    "Contact me at test@gmail.com",
    "😂😂😂😂😂",
    "1234567890"
]

<h2>Stress testing</h2>

In [19]:
print("STRESS TESTING RESULTS\n")

for s in sentences:
    tokens, clean = preprocess_text(s)

    print(f"""
Original Text   : {s}
Cleaned Tokens  : {tokens}
Cleaned Sentence: {clean}
{'='*60}
""")

STRESS TESTING RESULTS


Original Text   : Get 100% FREE access now!!!
Cleaned Tokens  : ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now


Original Text   : I absolutely looooved this product 😍😍
Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product


Original Text   : Worst service ever... 0/10
Cleaned Tokens  : ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever


Original Text   : Call me at 9876543210
Cleaned Tokens  : ['call']
Cleaned Sentence: call


Original Text   : This is THE best course!!!
Cleaned Tokens  : ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course


Original Text   : Visit https://openai.com now!
Cleaned Tokens  : ['visit', 'now']
Cleaned Sentence: visit now


Original Text   : Nooooo this is baaad!!!
Cleaned Tokens  : ['no', 'this', 'bad']
Cleaned Sentence: no this bad


Original Text   : OK OK OK I got it
Cleaned Tokens  : ['got']
Cleaned Sentence: got

<h2>Apply Preprocessing</h2>

In [12]:
results = []

for s in sentences:
    tokens, clean = preprocess_text(s)
    results.append((s, tokens, clean))

    print(f"""
Original Text   : {s}
Cleaned Tokens  : {tokens}
Cleaned Sentence: {clean}
{'-'*60}
""")


Original Text   : Get 100% FREE access now!!!
Cleaned Tokens  : ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
------------------------------------------------------------


Original Text   : I absolutely looooved this product 😍😍
Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product
------------------------------------------------------------


Original Text   : Worst service ever... 0/10
Cleaned Tokens  : ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
------------------------------------------------------------


Original Text   : Call me at 9876543210
Cleaned Tokens  : ['call']
Cleaned Sentence: call
------------------------------------------------------------


Original Text   : This is THE best course!!!
Cleaned Tokens  : ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
------------------------------------------------------------


Original Text   : Visit https

<h2>Token analytics</h2>

In [13]:
analytics = []

for original, tokens, clean in results:
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(w) for w in tokens) / total if total > 0 else 0

    analytics.append({
        "sentence": original,
        "total_tokens": total,
        "unique_tokens": unique,
        "avg_length": round(avg_len, 2)
    })

# Display nicely
for a in analytics:
    print(a)

{'sentence': 'Get 100% FREE access now!!!', 'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 4.0}
{'sentence': 'I absolutely looooved this product 😍😍', 'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 6.5}
{'sentence': 'Worst service ever... 0/10', 'total_tokens': 3, 'unique_tokens': 3, 'avg_length': 5.33}
{'sentence': 'Call me at 9876543210', 'total_tokens': 1, 'unique_tokens': 1, 'avg_length': 4.0}
{'sentence': 'This is THE best course!!!', 'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 4.25}
{'sentence': 'Visit https://openai.com now!', 'total_tokens': 2, 'unique_tokens': 2, 'avg_length': 4.0}
{'sentence': 'Nooooo this is baaad!!!', 'total_tokens': 3, 'unique_tokens': 3, 'avg_length': 3.0}
{'sentence': 'OK OK OK I got it', 'total_tokens': 1, 'unique_tokens': 1, 'avg_length': 3.0}
{'sentence': 'Win $$$ now!!! Limited offer!!!', 'total_tokens': 4, 'unique_tokens': 4, 'avg_length': 4.5}
{'sentence': 'I am not happy with this', 'total_tokens': 4, 'unique_tokens': 4, 'avg_l

<h2>Frequency analysis</h2>

In [14]:
all_tokens = []

for _, tokens, _ in results:
    all_tokens.extend(tokens)

freq = Counter(all_tokens)

print("Top 10 most frequent words:")
print(freq.most_common(10))

print("\nTop 5 least frequent words:")
print(freq.most_common()[-5:])

Top 10 most frequent words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 least frequent words:
[('offer', 1), ('not', 1), ('happy', 1), ('with', 1), ('contact', 1)]


<h2>Full pipeline function</h2>

In [15]:
def full_pipeline(text_list):
    if not isinstance(text_list, list):
        return {"tokens": [], "clean_sentences": []}

    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

<h2>Run full pipeline</h2>

In [16]:
output = full_pipeline(sentences)

print("All Tokens:\n", output["tokens"])
print("\nClean Sentences:\n", output["clean_sentences"])

All Tokens:
 ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this', 'contact']

Clean Sentences:
 ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited offer', 'not happy with this', 'contact', '', '']


<h2>Error handling testing</h2>

In [17]:
test_cases = ["", "😂😂😂", "123456"]

for t in test_cases:
    tokens, clean = preprocess_text(t)
    print(f"""
Input : {t}
Tokens: {tokens}
Clean : {clean}
{'-'*40}
""")


Input : 
Tokens: []
Clean : 
----------------------------------------


Input : 😂😂😂
Tokens: []
Clean : 
----------------------------------------


Input : 123456
Tokens: []
Clean : 
----------------------------------------



<h2>Required analysis questions</h2>

In [18]:
# Most noisy sentence (fewest tokens)
noisy = min(analytics, key=lambda x: x['total_tokens'])

# Most meaningful (most unique tokens)
meaningful = max(analytics, key=lambda x: x['unique_tokens'])

print("Most Noisy Sentence:")
print(noisy["sentence"])

print("\nMost Meaningful Sentence:")
print(meaningful["sentence"])

Most Noisy Sentence:
😂😂😂😂😂

Most Meaningful Sentence:
Get 100% FREE access now!!!
